# RBAC

ServiceAccount = Identität, Role = Rechte, RoleBinding = Zuordnung. Die Demonstration verwendet kubectl auth can-i mit Impersonation; dafür benötigt der Trainer entsprechende Rechte.

## Voraussetzungen

Python-Kernel, kubectl und K3s-Trainingscluster. Aus diesem Kapitelverzeichnis ausführen. Die verwendeten Demo-Ressourcen müssen frei sein. Aktiven Kontext und Namespace prüfen.

In [ ]:
%%bash
set -eo pipefail
kubectl config current-context
NS=$(kubectl config view --minify -o jsonpath='{..namespace}')
test "${NS:-default}" = default
kubectl get nodes

## 1. Namespace und Identität

Einen neuen Namespace erzeugen. Falls er existiert, abbrechen und nicht aufräumen.

In [ ]:
%%bash
set -eo pipefail
kubectl create namespace security-rbac-demo
kubectl -n security-rbac-demo create serviceaccount training-reader

## 2. Vorher prüfen

Ohne zusätzliche Bindungen sollte der Account keine Pods auflisten dürfen. Bestehende clusterweite Bindungen können das Ergebnis verändern.

In [ ]:
%%bash
set -eo pipefail
kubectl auth can-i list pods -n security-rbac-demo --as=system:serviceaccount:security-rbac-demo:training-reader || true

## 3. Rechte zuweisen

Nur lesender Zugriff auf Pods im Demo-Namespace.

In [ ]:
%%bash
set -eo pipefail
cat rbac.yml
kubectl apply -f rbac.yml
kubectl auth can-i list pods -n security-rbac-demo --as=system:serviceaccount:security-rbac-demo:training-reader
kubectl auth can-i delete pods -n security-rbac-demo --as=system:serviceaccount:security-rbac-demo:training-reader || true
kubectl auth can-i list secrets -n security-rbac-demo --as=system:serviceaccount:security-rbac-demo:training-reader || true

## 4. Aufräumen

Nur den neu angelegten Namespace löschen.

In [ ]:
%%bash
set -eo pipefail
kubectl delete namespace security-rbac-demo

## Fazit

RBAC vergibt Rechte an Identitäten; die effektiven Rechte ergeben sich aus allen Bindungen.